# Section 1 — Data layer & shared harness · Wai Chun

Working file for **Section 1** of `main.ipynb`: cleaning, aggregation to enrolment level,
the three feature sets, and the shared split / preprocess / evaluate harness that
Sections 2 and 3 both call.

Nothing is modelled here. The output is four wide tables (one per cutoff), a manifest,
and the `A2.*` helpers that `work_Guangyu` and `work_Xinyan` import.


## Plan & hints *(working notes — don't copy into `main.ipynb`)*

### Order of work — freeze Tue 14 Oct
1. 1.0–1.4 are already verified (Restart & Run All, Oct 1). Don't touch unless a number changes.
2. 1.5 is the new part: `a2_common.py` inlined, because **only one `.ipynb` may be submitted**
   and both other sections do `import a2_common`.
3. After merging into `main.ipynb`, Restart & Run All once end-to-end and check every number
   quoted in the report still matches.

### Rules
- `?` is the missing-value marker — always read with `na_values=["?"]`
- `date_unregistration` is **never** a feature; it only defines the risk set
- Features are truncated at the cutoff day; nothing after day *t* may enter a day-*t* table
- Activity-type columns are anchored to the full 20-type list from `vle.csv`, not to the
  types that happen to appear before the cutoff
- The three feature sets must satisfy `engineered == baseline + behaviour`, asserted in 1.4

### Feature sets handed to Sections 2 and 3
| Name | n | Definition |
|---|---|---|
| `baseline` | 14 | Known at registration (demographics + enrolment) |
| `behaviour` | 38 | Generated after enrolment (VLE clickstream + assignments) |
| `engineered` | 52 | `baseline` + `behaviour`, no overlap |

### Where each piece ends up in the report
| Notebook | Report section | Owner |
|---|---|---|
| 1.2 Evidence | 3.1 Dataset understanding | me |
| 1.3–1.4 | 3.2 Preprocessing & feature engineering | me |
| 1.5 `make_splits` | 3.3 Validation design | teammate |
| 1.5 `evaluate` | 3.5 Evaluation metrics | teammate |

### Open items
- [ ] Inline check: after merging, confirm `A2.` prefix still resolves for Sections 2–3
- [ ] `sum_click` outlier handling (max 6,977 vs mean 3.72) — log vs winsorise
- [ ] Within-module comparison of repeat vs first-attempt students (fixed module effect)


## 1.0 Config

Paths, seed, key, and the four cutoff days.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
OUT_DIR = Path("processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
KEY = ["code_module", "code_presentation", "id_student"]

# Prediction points: course start, first assignment due, second assignment due, midpoint.
CUTOFF_DAYS = [0, 24, 60, 120]

# Earliest date in studentVle; used as the left edge of the aggregation window.
VLE_WINDOW_START = -25

## 1.1 Load raw tables

Seven CSVs. `studentVle` is 10.7 M rows, so dtypes are specified explicitly.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# Missing values are encoded as the string "?", not as empty fields.
NA = ["?"]

student_info = pd.read_csv(DATA_DIR / "studentInfo.csv", na_values=NA)
student_reg = pd.read_csv(DATA_DIR / "studentRegistration.csv", na_values=NA)
student_asm = pd.read_csv(DATA_DIR / "studentAssessment.csv", na_values=NA)
assessments = pd.read_csv(DATA_DIR / "assessments.csv", na_values=NA)
vle_meta = pd.read_csv(DATA_DIR / "vle.csv", na_values=NA)
courses = pd.read_csv(DATA_DIR / "courses.csv", na_values=NA)

In [ ]:
# studentVle is 10.7M rows; explicit dtypes cut memory from ~1.5 GB to ~250 MB.
VLE_DTYPES = {
    "code_module": "category",
    "code_presentation": "category",
    "id_student": "int32",
    "id_site": "int32",
    "date": "int16",
    "sum_click": "int32",
}

student_vle = pd.read_csv(DATA_DIR / "studentVle.csv", dtype=VLE_DTYPES, na_values=NA)
student_vle.shape

## 1.2 Evidence for the report

Three non-obvious properties of the dataset, each quantified here so Section 3.1 can cite a
number rather than assert a claim. **Nothing is modified in this subsection** — it only measures.

### 1.2.1 Missing values are encoded as the string `?`

Read with defaults, the affected columns parse as strings and missingness is reported as 0%.

In [ ]:
# Reading each file twice, with and without na_values, shows how the "?" marker
# changes both the reported missing rate and the inferred dtype.
rows = []
for name in ["studentInfo", "studentRegistration", "studentAssessment", "vle"]:
    raw = pd.read_csv(DATA_DIR / f"{name}.csv")
    fixed = pd.read_csv(DATA_DIR / f"{name}.csv", na_values=NA)
    for col in raw.columns:
        n = (raw[col].astype(str).str.strip() == "?").sum()
        if n:
            rows.append({"table": name, "column": col, "n_question_marks": n,
                         "share_%": round(n / len(raw) * 100, 2),
                         "dtype_default": str(raw[col].dtype),
                         "dtype_corrected": str(fixed[col].dtype)})

evidence_1 = pd.DataFrame(rows)
evidence_1.to_csv(OUT_DIR / "evidence_1_hidden_missing.csv", index=False)
evidence_1

### 1.2.2 `date_unregistration` is target leakage

Populated for almost every Withdrawn record and for none of the Pass / Distinction records.

In [ ]:
chk = student_info.merge(student_reg, on=KEY, how="left")

evidence_2 = pd.crosstab(chk["final_result"],
                         chk["date_unregistration"].isna(),
                         normalize="index").round(4) * 100
evidence_2.columns = ["has_unregistration_date_%", "no_unregistration_date_%"]
evidence_2.to_csv(OUT_DIR / "evidence_2_leakage.csv")
evidence_2

### 1.2.3 Grouped structure — why a random split would not be valid

Quantifies how strongly a student's enrolments depend on each other. Feeds Section 3.3.

In [ ]:
vc = student_info["id_student"].value_counts()

si = student_info.copy()
si["completed"] = si["final_result"].isin(["Pass", "Distinction"])

# Order presentations chronologically so "first" and "second" attempt are well defined.
ORDER = {"2013B": 0, "2013J": 1, "2014B": 2, "2014J": 3}
multi = (si[si["id_student"].isin(vc[vc > 1].index)]
         .assign(_o=lambda d: d["code_presentation"].map(ORDER))
         .sort_values(["id_student", "_o"]))
pairs = multi.groupby("id_student")["completed"].apply(list)
pairs = pairs[pairs.apply(len) >= 2]
first, second = pairs.apply(lambda x: x[0]), pairs.apply(lambda x: x[1])

evidence_3 = pd.DataFrame([
    {"quantity": "enrolments", "value": len(student_info)},
    {"quantity": "unique students", "value": student_info["id_student"].nunique()},
    {"quantity": "students with >1 enrolment", "value": int((vc > 1).sum())},
    {"quantity": "max enrolments per student", "value": int(vc.max())},
    {"quantity": "completion rate, overall %", "value": round(si["completed"].mean() * 100, 1)},
    {"quantity": "completion rate | first attempt completed %", "value": round(second[first].mean() * 100, 1)},
    {"quantity": "completion rate | first attempt failed %", "value": round(second[~first].mean() * 100, 1)},
])
evidence_3.to_csv(OUT_DIR / "evidence_3_grouping.csv", index=False)
evidence_3

## 1.3 Clean `studentInfo` and construct the target

Binary target: Fail + Withdrawn → `at_risk = 1`; Pass + Distinction → `0`.
Also fixes the `imd_band` formatting bug and merges registration information.

In [ ]:
info = student_info.copy()

# Positive class is non-completion, so precision and recall map onto the two
# costs of an early-warning system.
info["at_risk"] = (~info["final_result"].isin(["Pass", "Distinction"])).astype("int8")

info["final_result"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
# The "10-20" band is missing its percent sign in 3,516 records; left unfixed,
# one-hot encoding would split one band into two categories.
info["imd_band"] = (info["imd_band"].astype("string").str.strip()
                    .str.replace(r"(?<!%)$", "%", regex=True))

# imd_band, highest_education and age_band are ordered, so they are encoded as
# ranks rather than one-hot, which would discard the ordering.
IMD_MAP = {f"{i}-{i+10}%": i + 5 for i in range(0, 100, 10)}
IMD_MAP["0-10%"] = 5
info["imd_band_num"] = info["imd_band"].map(IMD_MAP).astype("float32")

EDU_MAP = {"No Formal quals": 0, "Lower Than A Level": 1, "A Level or Equivalent": 2,
           "HE Qualification": 3, "Post Graduate Qualification": 4}
info["highest_education_ord"] = info["highest_education"].map(EDU_MAP).astype("float32")

AGE_MAP = {"0-35": 0, "35-55": 1, "55<=": 2}
info["age_band_ord"] = info["age_band"].map(AGE_MAP).astype("float32")

# Only imd_band carries genuine missingness; the other two must map cleanly.
assert info["highest_education_ord"].isna().sum() == 0
assert info["age_band_ord"].isna().sum() == 0

In [ ]:
info = info.merge(student_reg, on=KEY, how="left", validate="one_to_one")

# Negative date_registration means days before course start; flipped so larger = earlier.
info["reg_lead_days"] = -info["date_registration"]
info["reg_date_missing"] = info["date_registration"].isna().astype("int8")

# 234 records (0.72%) register after the course starts; flagged rather than dropped.
info["registered_after_start"] = (info["date_registration"] > 0).fillna(False).astype("int8")

# Course length varies from 234 to 269 days, so day counts are later turned into ratios.
info = info.merge(courses, on=["code_module", "code_presentation"], how="left")

assert len(info) == len(student_info)

## 1.4 Aggregate to enrolment level and define the feature sets

One row per enrolment — the triple (module, presentation, student) — per cutoff day.
The risk set and the cutoff truncation are both applied here.

### 1.4.1 Prepare VLE and assessment sources

In [ ]:
assert vle_meta["id_site"].is_unique
site2type = vle_meta.set_index("id_site")["activity_type"]

# Anchored to the full type list from vle.csv: types such as repeatactivity are not
# clicked before the early cutoffs, so deriving columns from observed types would
# give the four cutoffs different feature schemas.
ALL_ACTIVITY_TYPES = sorted(vle_meta["activity_type"].dropna().unique().tolist())

student_vle["activity_type"] = student_vle["id_site"].map(site2type).astype("category")
assert student_vle["activity_type"].isna().sum() == 0

len(ALL_ACTIVITY_TYPES)

In [ ]:
# studentAssessment carries no module or presentation; both arrive via assessments.
asm = student_asm.merge(
    assessments[["id_assessment", "code_module", "code_presentation",
                 "assessment_type", "date", "weight"]],
    on="id_assessment", how="left", validate="many_to_one")

asm = asm.rename(columns={"date": "due_date"})
asm["days_early"] = asm["due_date"] - asm["date_submitted"]
asm.shape

### 1.4.2 Aggregation functions

In [ ]:
def _normalise_key(f):
    """Cast the key to str/int64 so category-dtype indexes join correctly."""
    f = f.reset_index()
    f["code_module"] = f["code_module"].astype(str)
    f["code_presentation"] = f["code_presentation"].astype(str)
    f["id_student"] = f["id_student"].astype("int64")
    return f.set_index(KEY)


def vle_features(svle, cutoff):
    """Aggregate clickstream to one row per enrolment, using days <= cutoff only."""
    w = svle[svle["date"] <= cutoff]
    if len(w) == 0:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=KEY))

    g = w.groupby(KEY, observed=True)
    f = g.agg(
        vle_total_clicks=("sum_click", "sum"),
        vle_active_days=("date", "nunique"),
        vle_n_sites=("id_site", "nunique"),
        vle_max_clicks_day=("sum_click", "max"),
    )
    f["vle_mean_clicks_per_active_day"] = f["vle_total_clicks"] / f["vle_active_days"]

    # Trend: a declining trajectory should signal disengagement more than a flat low one.
    mid = (VLE_WINDOW_START + cutoff) / 2
    h1 = w[w["date"] <= mid].groupby(KEY, observed=True)["sum_click"].sum()
    h2 = w[w["date"] > mid].groupby(KEY, observed=True)["sum_click"].sum()
    f["vle_clicks_h1"] = h1.reindex(f.index).fillna(0)
    f["vle_clicks_h2"] = h2.reindex(f.index).fillna(0)
    f["vle_trend_ratio"] = (f["vle_clicks_h2"] + 1) / (f["vle_clicks_h1"] + 1)

    f["vle_clicks_before_start"] = (w[w["date"] <= 0].groupby(KEY, observed=True)["sum_click"]
                                    .sum().reindex(f.index).fillna(0))

    # Composition rather than volume: share of clicks per resource type.
    at = (w.groupby(KEY + ["activity_type"], observed=True)["sum_click"]
            .sum().unstack(fill_value=0))
    at = at.div(at.sum(axis=1).replace(0, np.nan), axis=0)
    at = at.reindex(columns=ALL_ACTIVITY_TYPES, fill_value=0.0)
    at.columns = ["vle_prop_" + str(col) for col in at.columns]

    return _normalise_key(f.join(at))


def asm_features(asm_df, cutoff):
    """Aggregate submissions to one row per enrolment, using submissions <= cutoff only."""
    w = asm_df[asm_df["date_submitted"] <= cutoff].copy()
    if len(w) == 0:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=KEY))

    w["_wscore"] = w["score"] * w["weight"]
    f = w.groupby(KEY, observed=True).agg(
        asm_n_submitted=("id_assessment", "nunique"),
        asm_mean_score=("score", "mean"),
        asm_min_score=("score", "min"),
        asm_n_banked=("is_banked", "sum"),
        asm_mean_days_early=("days_early", "mean"),
        _wsum=("_wscore", "sum"),
        _wtot=("weight", "sum"),
    )
    f["asm_weighted_score"] = f["_wsum"] / f["_wtot"].replace(0, np.nan)
    return _normalise_key(f.drop(columns=["_wsum", "_wtot"]))

### 1.4.3 Risk set and wide-table builder

Students who had already withdrawn before day *t* are excluded from the day-*t* table: for them the outcome has already occurred, so including them would be recording rather than predicting.

In [ ]:
def build_wide_table(cutoff):
    """Build the day-`cutoff` wide table: risk set, truncated features, one row per enrolment."""
    base = info.copy()

    # Risk set: a student who withdrew on or before the cutoff already has an outcome,
    # so including them would be recording rather than predicting.
    already_withdrawn = base["date_unregistration"].notna() & (base["date_unregistration"] <= cutoff)
    n_excluded = int(already_withdrawn.sum())
    base = base.loc[~already_withdrawn].copy()

    vf = vle_features(student_vle, cutoff)
    af = asm_features(asm, cutoff)

    out = (base.set_index(KEY).join(vf, how="left").join(af, how="left").reset_index())

    vle_cols = [c for c in out.columns if c.startswith("vle_")]

    # Membership of the aggregate index, so the flags do not depend on column order.
    idx = pd.MultiIndex.from_frame(out[KEY])
    out["has_vle_activity"] = idx.isin(vf.index).astype("int8")
    out["has_asm_activity"] = idx.isin(af.index).astype("int8")
    n_zero_vle = int((out["has_vle_activity"] == 0).sum())

    # Count features impute to zero because no activity genuinely means zero;
    # proportions and means stay NaN so the modelling imputer handles them.
    count_like = [c for c in vle_cols if not any(k in c for k in ("prop", "trend", "mean"))]
    out[count_like] = out[count_like].fillna(0)
    out[["asm_n_submitted", "asm_n_banked"]] = out[["asm_n_submitted", "asm_n_banked"]].fillna(0)

    # Active days as a ratio, since the aggregation window length varies by cutoff.
    out["vle_active_days_ratio"] = out["vle_active_days"] / (cutoff - VLE_WINDOW_START + 1)
    out["cutoff_day"] = cutoff

    return out, dict(cutoff=cutoff, n_excluded=n_excluded, n=len(out),
                     n_students=out["id_student"].nunique(),
                     zero_vle=n_zero_vle, pos_rate=round(out["at_risk"].mean() * 100, 2))

### 1.4.4 Build all four cutoffs

In [ ]:
%%time
tables, summary = {}, []
for cut in CUTOFF_DAYS:
    t, s = build_wide_table(cut)
    tables[cut] = t
    summary.append(s)

summary = pd.DataFrame(summary)
summary.to_csv(OUT_DIR / "cutoff_summary.csv", index=False)
summary

### 1.4.5 The three feature sets

`engineered == baseline + behaviour` is asserted, so the sets cannot silently drift apart.

In [ ]:
sample = tables[CUTOFF_DAYS[1]]

# Known at registration.
BACKGROUND_CAT = ["code_module", "code_presentation", "gender", "region", "disability"]
BACKGROUND_NUM = ["highest_education_ord", "imd_band_num", "age_band_ord",
                  "num_of_prev_attempts", "studied_credits",
                  "reg_lead_days", "reg_date_missing", "registered_after_start",
                  "module_presentation_length"]

# Generated after enrolment.
BEHAVIOUR_NUM = [c for c in sample.columns if c.startswith(("vle_", "asm_"))] + \
                ["has_vle_activity", "has_asm_activity"]

# Never usable as features: the target itself, the leaking column, raw forms of
# already-encoded variables, the identifier, and the cutoff marker.
FORBIDDEN = ["final_result", "at_risk", "date_unregistration", "date_registration",
             "imd_band", "highest_education", "age_band", "id_student", "cutoff_day"]

FEATURE_SETS = {
    "baseline": BACKGROUND_CAT + BACKGROUND_NUM,
    "behaviour": BEHAVIOUR_NUM,
    "engineered": BACKGROUND_CAT + BACKGROUND_NUM + BEHAVIOUR_NUM,
}

# Guards so the three sets cannot drift apart if anyone edits the lists above.
assert set(FEATURE_SETS["engineered"]) == set(FEATURE_SETS["baseline"]) | set(FEATURE_SETS["behaviour"])
assert not (set(FEATURE_SETS["baseline"]) & set(FEATURE_SETS["behaviour"]))
for name, cols in FEATURE_SETS.items():
    assert all(c in sample.columns for c in cols)
    assert not any(c in FORBIDDEN for c in cols)

{name: len(cols) for name, cols in FEATURE_SETS.items()}

### 1.4.6 Skew diagnosis → `SKEWED_COLS`

Selection rule: skew > 2 **and** log reduces |skew| by ≥ 50% **and** max > 10 — so that 0–1 proportion columns, whose skew comes from sparsity rather than a long tail, are not selected.

In [ ]:
rows = []
for col in [c for c in FEATURE_SETS["engineered"] if c not in BACKGROUND_CAT]:
    s = sample[col].dropna()
    if s.nunique() <= 2 or s.min() < 0:
        continue
    sk, skl = s.skew(), np.log1p(s).skew()
    rows.append({"column": col, "skew": round(sk, 2), "skew_after_log": round(skl, 2),
                 "reduction_%": round((abs(sk) - abs(skl)) / abs(sk) * 100, 1) if sk else 0.0,
                 "max": round(s.max(), 2)})

skew_table = pd.DataFrame(rows).sort_values("skew", ascending=False)

# Selected only if heavily skewed AND log halves the skew AND the column is a magnitude:
# the vle_prop_* columns are skewed through sparsity, not a long tail, so log does nothing.
selected = skew_table[(skew_table["skew"] > 2) &
                      (skew_table["reduction_%"] >= 50) &
                      (skew_table["max"] > 10)]
SKEWED_COLS = selected["column"].tolist()

skew_table.to_csv(OUT_DIR / "evidence_4_skewness.csv", index=False)
selected.sort_values("reduction_%", ascending=False)

## 1.5 Shared harness

This is `a2_common.py` inlined. **Only one `.ipynb` may be submitted**, so the module cannot
ship as a separate file — but Sections 2 and 3 both call `A2.make_splits`, `A2.feature_cols`
and `A2.score_model`.

Running this subsection defines those names at top level *and* exposes them under the alias
`A2`, so existing `A2.xxx` calls in Sections 2 and 3 keep working unchanged.

### 1.5.1 Constants, manifest and feature sets

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

RANDOM_SEED = SEED
PROCESSED_DIR = OUT_DIR
GROUP_COL = "id_student"
TARGET = "at_risk"
MAIN_CUTOFF = 24
FEATURE_SET_NAMES = ("baseline", "behaviour", "engineered")

_manifest = None


def manifest():
    """Load and cache manifest.json."""
    global _manifest
    if _manifest is None:
        with open(PROCESSED_DIR / "manifest.json", encoding="utf-8") as fh:
            _manifest = json.load(fh)
    return _manifest


def feature_cols(which="engineered"):
    """Column names for one feature set; engineered = baseline + behaviour."""
    sets = manifest()["feature_sets"]
    if which == "behaviour" and "behaviour" not in sets:
        base = set(sets["baseline"])
        return [c for c in sets["engineered"] if c not in base]
    if which not in sets:
        raise KeyError(f"unknown feature set {which!r}; available: {', '.join(sets)}")
    return list(sets[which])


def load_table(cutoff=MAIN_CUTOFF):
    """Read the wide table for one cutoff day."""
    path = PROCESSED_DIR / f"wide_cutoff_{cutoff:03d}.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found; run Section 1.6 first")
    return pd.read_csv(path)

### 1.5.2 Splitting — grouped by student, stratified on the target

The grouping is what the evidence in 1.2.3 requires. Asserts that no student crosses a split.

In [ ]:
def make_splits(df, seed=SEED):
    """Split 60/20/20, grouped by student and stratified on the target."""
    y, g = df[TARGET], df[GROUP_COL]

    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trainval_pos, test_pos = next(outer.split(df, y, g))
    trainval = df.iloc[trainval_pos].reset_index(drop=True)
    test = df.iloc[test_pos].reset_index(drop=True)

    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_pos, val_pos = next(inner.split(trainval, trainval[TARGET], trainval[GROUP_COL]))
    train = trainval.iloc[tr_pos].reset_index(drop=True)
    val = trainval.iloc[val_pos].reset_index(drop=True)

    # No student may appear in more than one split.
    s_tr, s_va, s_te = (set(d[GROUP_COL]) for d in (train, val, test))
    assert not (s_tr & s_va) and not (s_tr & s_te) and not (s_va & s_te)

    return train, val, test


def describe_splits(train, val, test):
    """Split sizes and class balance, as reported in the Method section."""
    total = len(train) + len(val) + len(test)
    return pd.DataFrame([{
        "split": name,
        "n": len(d),
        "share_%": round(len(d) / total * 100, 1),
        "students": d[GROUP_COL].nunique(),
        "positive_%": round(d[TARGET].mean() * 100, 2),
    } for name, d in [("train", train), ("validation", val), ("test", test)]])

### 1.5.3 Preprocessing

Fitted inside a `Pipeline` on training data only — fitting a scaler on the full dataset before splitting leaks test statistics into training.

In [ ]:
def _log1p_nonneg(X):
    """log(1 + x) with negatives clipped; named rather than a lambda so it pickles."""
    return np.log1p(np.clip(X, 0, None))


def make_preprocessor(cols, log_transform=False):
    """Impute, optionally log-transform skewed columns, scale numerics, one-hot categoricals."""
    mf = manifest()
    cat = [c for c in cols if c in mf["categorical_features"]]
    num = [c for c in cols if c not in cat]

    skewed = [c for c in num if c in mf.get("skewed_cols", [])] if log_transform else []
    plain = [c for c in num if c not in skewed]

    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
    skew_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("log", FunctionTransformer(_log1p_nonneg, feature_names_out="one-to-one")),
                          ("scale", StandardScaler())])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

    transformers = [("num", num_pipe, plain)]
    if skewed:
        transformers.append(("skew", skew_pipe, skewed))
    if cat:
        transformers.append(("cat", cat_pipe, cat))

    return ColumnTransformer(transformers, remainder="drop")

### 1.5.4 Metrics

In [ ]:
def evaluate(y_true, y_pred, y_proba=None):
    """Metric set used for every model; the positive class is non-completion."""
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    if y_proba is not None:
        out["roc_auc"] = roc_auc_score(y_true, y_proba)
    return {k: round(float(v), 4) for k, v in out.items()}


def confusion(y_true, y_pred):
    """Confusion matrix with readable row and column labels."""
    return pd.DataFrame(confusion_matrix(y_true, y_pred),
                        index=["true: completed", "true: at risk"],
                        columns=["pred: completed", "pred: at risk"])


def score_model(model, cols, train, val, log_transform=False, fit_on=None):
    """Build the pipeline, fit it, and evaluate on `val`. Returns (pipeline, metrics)."""
    pipe = Pipeline([("prep", make_preprocessor(cols, log_transform)), ("clf", model)])
    fit_df = train if fit_on is None else fit_on
    pipe.fit(fit_df[cols], fit_df[TARGET])
    pred = pipe.predict(val[cols])
    proba = pipe.predict_proba(val[cols])[:, 1] if hasattr(pipe, "predict_proba") else None
    return pipe, evaluate(val[TARGET], pred, proba)

### 1.5.5 Alias + smoke test

Makes `A2.` resolve inside one notebook, then checks the harness end to end.

In [ ]:
import sys
import types

# Expose the names above under the alias `A2` and register them as the module
# `a2_common`, so the `import a2_common as A2` lines in Sections 2 and 3 resolve
# without a separate file.
A2 = types.ModuleType("a2_common")
A2.__dict__.update({k: v for k, v in globals().items() if not k.startswith("__")})
sys.modules["a2_common"] = A2

_df = A2.load_table(A2.MAIN_CUTOFF)
_tr, _va, _te = A2.make_splits(_df)
assert {n: len(A2.feature_cols(n)) for n in A2.FEATURE_SET_NAMES} == {
    "baseline": 14, "behaviour": 38, "engineered": 52}

A2.describe_splits(_tr, _va, _te)

## 1.6 Self-check and delivery

Guards that must pass before the tables are handed to the modelling sections.

In [ ]:
t = tables[CUTOFF_DAYS[1]]

assert t.groupby(KEY).ngroups == len(t)
assert t["at_risk"].isna().sum() == 0
assert set(t["at_risk"].unique()) <= {0, 1}
assert all(c not in FEATURE_SETS["engineered"] for c in FORBIDDEN)
assert "date_unregistration" in t.columns
assert (t["has_vle_activity"] == 0).sum() > 0
assert len(tables) == len(CUTOFF_DAYS)

# Identical column sets across cutoffs is what makes the across-cutoff comparison valid.
assert all(list(tables[c].columns) == list(t.columns) for c in CUTOFF_DAYS)

(t[FEATURE_SETS["engineered"]].isna().mean().mul(100).round(2)
 .sort_values(ascending=False).head(10).rename("missing_%"))

### 1.6.1 Save wide tables + manifest

In [ ]:
import json

for cut, t in tables.items():
    path = OUT_DIR / f"wide_cutoff_{cut:03d}.csv"
    t.to_csv(path, index=False)
    print(f"{path.name:24s} {t.shape}")

manifest = {
    "random_seed": SEED,
    "key": KEY,
    "group_column": "id_student",
    "target": "at_risk",
    "target_definition": "1 = Fail or Withdrawn; 0 = Pass or Distinction",
    "cutoffs": CUTOFF_DAYS,
    "feature_sets": FEATURE_SETS,
    "categorical_features": BACKGROUND_CAT,
    "forbidden_columns": FORBIDDEN,
    "skewed_cols": SKEWED_COLS,
    "notes": [
        "Read every CSV with na_values=['?']",
        "date_unregistration defines the risk set and must never be a feature",
        "Splits must be grouped by id_student",
        "Preprocessing must sit inside a Pipeline and be fitted on training data only",
    ],
}
with open(OUT_DIR / "manifest.json", "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

print(f"{'manifest.json':24s} {len(FEATURE_SETS)} feature sets")